<a href="https://colab.research.google.com/github/AdAdalan/NLP-Assignment3/blob/main/54__COMP90042_Project_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [7]:
# ============================================================
# Install packages and mount Google Drive
# ============================================================

import os
from pathlib import Path

%cd /content

if not os.path.exists('/content/NLP-Assignment3'):
    !git clone https://github.com/AdAdalan/NLP-Assignment3.git
else:
    print('Repo already exists.')

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers sentence-transformers bm25s

/content
Repo already exists.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ============================================================
# Global paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/NLP ASS3')
DATA_DIR = PROJECT_ROOT / 'data'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)

PROJECT_ROOT: /content/drive/MyDrive/NLP ASS3
DATA_DIR: /content/drive/MyDrive/NLP ASS3/data


In [9]:
# ============================================================
# Load data
# Same loading logic as bge2.ipynb
# ============================================================

import json

with open(DATA_DIR / 'train-claims.json', 'r') as f:
    train_claims = json.load(f)

with open(DATA_DIR / 'dev-claims.json', 'r') as f:
    dev_claims = json.load(f)

with open(DATA_DIR / 'test-claims-unlabelled.json', 'r') as f:
    test_claims = json.load(f)

with open(DATA_DIR / 'evidence.json', 'r') as f:
    evidence = json.load(f)

train_ids = list(train_claims.keys())
dev_ids = list(dev_claims.keys())
test_ids = list(test_claims.keys())
evidence_ids = list(evidence.keys())

train_texts_raw = [train_claims[cid]['claim_text'] for cid in train_ids]
dev_texts_raw = [dev_claims[cid]['claim_text'] for cid in dev_ids]
test_texts_raw = [test_claims[cid]['claim_text'] for cid in test_ids]
evidence_texts = [evidence[eid] for eid in evidence_ids]

print('Train claims:', len(train_claims))
print('Dev claims:', len(dev_claims))
print('Test claims:', len(test_claims))
print('Evidence passages:', len(evidence))

Train claims: 1228
Dev claims: 154
Test claims: 153
Evidence passages: 1208827


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Soft prompt + Hard Prefix Prompt + pretrained BGE fine-tuned + BM25S

In [10]:
# ============================================================
# Seed and device
# ============================================================

import random
import numpy as np
import torch
from transformers import set_seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Seed:', SEED)

Device: cuda
Seed: 42


In [11]:
# ============================================================
# Shared constants and retrieval/evaluation helpers
# ============================================================

import json
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

BGE_MODEL_NAME = 'BAAI/bge-small-en-v1.5'
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

def build_query_inputs(raw_texts, use_prefix=True):
    if use_prefix:
        return [BGE_QUERY_PREFIX + text for text in raw_texts]
    return list(raw_texts)


@torch.no_grad()
def encode_bge_texts(
    texts,
    tokenizer,
    model,
    batch_size=64,
    max_len=256,
    desc='Encoding'
):
    model.eval()
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}

        outputs = model(**encoded)
        embeddings = outputs.last_hidden_state[:, 0]
        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.detach().cpu())

    return torch.cat(all_embeddings, dim=0)


def dense_retrieve_from_embeddings(
    claim_ids,
    claim_emb,
    evidence_ids,
    evidence_emb,
    top_k=100,
    batch_size=64
):
    predictions = {}

    claim_emb = claim_emb.to(device)
    evidence_emb = evidence_emb.to(device)
    evidence_emb_t = evidence_emb.T

    top_k = min(top_k, len(evidence_ids))

    for start in tqdm(
        range(0, len(claim_ids), batch_size),
        desc=f'Dense retrieving top-{top_k}'
    ):
        end = start + batch_size
        batch_claim_emb = claim_emb[start:end]

        scores = batch_claim_emb @ evidence_emb_t
        _, top_indices = torch.topk(scores, k=top_k, dim=1)
        top_indices = top_indices.detach().cpu().numpy()

        for row_idx, claim_id in enumerate(claim_ids[start:end]):
            predictions[claim_id] = [
                evidence_ids[index]
                for index in top_indices[row_idx]
            ]

    return predictions


def eval_retrieval(claims_dataset, predictions):
    recalls, precisions, fscores = [], [], []

    for claim_id, claim in sorted(claims_dataset.items()):
        true_evidence_ids = claim.get('evidences', [])
        predicted_evidence_ids = predictions.get(claim_id, [])

        if len(true_evidence_ids) == 0 or len(predicted_evidence_ids) == 0:
            recalls.append(0.0)
            precisions.append(0.0)
            fscores.append(0.0)
            continue

        true_set = set(true_evidence_ids)
        pred_set = set(predicted_evidence_ids)
        correct = len(true_set.intersection(pred_set))

        recall = correct / len(true_set)
        precision = correct / len(predicted_evidence_ids)
        fscore = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

        recalls.append(recall)
        precisions.append(precision)
        fscores.append(fscore)

    metrics = {
        'mean_recall': float(np.mean(recalls)),
        'mean_precision': float(np.mean(precisions)),
        'mean_f1': float(np.mean(fscores)),
        'min_recall': float(np.min(recalls)),
    }

    print(json.dumps(metrics, indent=2))
    return metrics


def union_candidate_lists(*candidate_dicts, max_candidates=None):
    merged = {}

    all_claim_ids = set()
    for candidate_dict in candidate_dicts:
        all_claim_ids.update(candidate_dict.keys())

    for claim_id in all_claim_ids:
        seen = set()
        merged_list = []

        for candidate_dict in candidate_dicts:
            for evidence_id in candidate_dict.get(claim_id, []):
                if evidence_id not in seen:
                    seen.add(evidence_id)
                    merged_list.append(evidence_id)

                if max_candidates is not None and len(merged_list) >= max_candidates:
                    break

            if max_candidates is not None and len(merged_list) >= max_candidates:
                break

        merged[claim_id] = merged_list

    return merged


print('Helper functions ready.')

Helper functions ready.


In [12]:
# ============================================================
# Training pairs for BGE / soft prompt contrastive learning
# ============================================================

from torch.utils.data import Dataset, DataLoader

def build_train_pairs(train_claims, evidence, use_query_prefix=True):
    pairs = []

    for claim_id, claim in train_claims.items():
        query_text = claim['claim_text']
        if use_query_prefix:
            query_text = BGE_QUERY_PREFIX + query_text

        for evidence_id in claim.get('evidences', []):
            if evidence_id in evidence:
                pairs.append({
                    'claim_id': claim_id,
                    'evidence_id': evidence_id,
                    'query': query_text,
                    'positive': evidence[evidence_id],
                })

    return pairs


class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]


def pair_collate_fn(batch):
    return {
        'queries': [item['query'] for item in batch],
        'positives': [item['positive'] for item in batch],
    }


def make_pair_loader(pairs, batch_size=16, shuffle=True, seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        PairDataset(pairs),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=pair_collate_fn,
        generator=generator,
    )


def in_batch_contrastive_loss(query_emb, positive_emb, temperature=0.05):
    logits = (query_emb @ positive_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


train_pairs_with_prefix = build_train_pairs(
    train_claims=train_claims,
    evidence=evidence,
    use_query_prefix=True
)

print('Number of train query-positive pairs:', len(train_pairs_with_prefix))
print(train_pairs_with_prefix[0])

Number of train query-positive pairs: 4122
{'claim_id': 'claim-1937', 'evidence_id': 'evidence-442946', 'query': 'Represent this sentence for searching relevant passages: Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.', 'positive': 'At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.'}


In [13]:
# ============================================================
# 2.1 Load pretrained BGE for fine-tuning
# ============================================================

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

hardprefix_bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME)
hardprefix_bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME).to(device)

print('Loaded pretrained BGE:', BGE_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pretrained BGE: BAAI/bge-small-en-v1.5


In [14]:
# ============================================================
# 2.2 Fine-tuning encoding function
# ============================================================

def encode_for_finetuning(texts, tokenizer, model, max_len=256):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}

    outputs = model(**encoded)
    embeddings = outputs.last_hidden_state[:, 0]
    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [15]:
# ============================================================
# 2.3 Hard Prefix Prompt + pretrained BGE fine-tuning
# Train set only
# Top-5 is used later as the monitoring retrieval k
# ============================================================

BGE_FINETUNE_BATCH_SIZE = 16
BGE_FINETUNE_EPOCHS = 3
BGE_FINETUNE_LR = 2e-5
BGE_FINETUNE_WEIGHT_DECAY = 0.01
BGE_FINETUNE_TEMPERATURE = 0.05

hardprefix_bge_train_loader = make_pair_loader(
    train_pairs_with_prefix,
    batch_size=BGE_FINETUNE_BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

hardprefix_bge_optimizer = AdamW(
    hardprefix_bge_model.parameters(),
    lr=BGE_FINETUNE_LR,
    weight_decay=BGE_FINETUNE_WEIGHT_DECAY
)

num_training_steps = len(hardprefix_bge_train_loader) * BGE_FINETUNE_EPOCHS
num_warmup_steps = int(0.1 * num_training_steps)

hardprefix_bge_scheduler = get_linear_schedule_with_warmup(
    hardprefix_bge_optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

for epoch in range(BGE_FINETUNE_EPOCHS):
    hardprefix_bge_model.train()
    epoch_losses = []

    for batch in tqdm(
        hardprefix_bge_train_loader,
        desc=f'Hard-prefix BGE fine-tuning epoch {epoch + 1}/{BGE_FINETUNE_EPOCHS}'
    ):
        query_emb = encode_for_finetuning(
            batch['queries'],
            hardprefix_bge_tokenizer,
            hardprefix_bge_model,
            max_len=256
        )

        positive_emb = encode_for_finetuning(
            batch['positives'],
            hardprefix_bge_tokenizer,
            hardprefix_bge_model,
            max_len=512
        )

        loss = in_batch_contrastive_loss(
            query_emb,
            positive_emb,
            temperature=BGE_FINETUNE_TEMPERATURE
        )

        hardprefix_bge_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(hardprefix_bge_model.parameters(), max_norm=1.0)
        hardprefix_bge_optimizer.step()
        hardprefix_bge_scheduler.step()

        epoch_losses.append(loss.item())

    print(f'Epoch {epoch + 1} mean loss: {np.mean(epoch_losses):.6f}')

# Important variable names:
# hardprefix_bge_model
# hardprefix_bge_tokenizer

Hard-prefix BGE fine-tuning epoch 1/3: 100%|██████████| 258/258 [00:55<00:00,  4.64it/s]


Epoch 1 mean loss: 0.718285


Hard-prefix BGE fine-tuning epoch 2/3: 100%|██████████| 258/258 [00:36<00:00,  7.11it/s]


Epoch 2 mean loss: 0.445183


Hard-prefix BGE fine-tuning epoch 3/3: 100%|██████████| 258/258 [00:39<00:00,  6.52it/s]

Epoch 3 mean loss: 0.372132


In [16]:
# ============================================================
# 2.4 Encode all sets with hard-prefix fine-tuned BGE
# Variables are kept in memory only
# ============================================================

hardprefix_bge_model.eval()

hardprefix_finetuned_bge_evidence_emb = encode_bge_texts(
    evidence_texts,
    hardprefix_bge_tokenizer,
    hardprefix_bge_model,
    batch_size=64,
    max_len=512,
    desc='Encoding evidence with hardprefix_finetuned_bge'
)

hardprefix_finetuned_bge_train_emb = encode_bge_texts(
    build_query_inputs(train_texts_raw, use_prefix=True),
    hardprefix_bge_tokenizer,
    hardprefix_bge_model,
    batch_size=64,
    max_len=256,
    desc='Encoding train claims with hardprefix_finetuned_bge'
)

hardprefix_finetuned_bge_dev_emb = encode_bge_texts(
    build_query_inputs(dev_texts_raw, use_prefix=True),
    hardprefix_bge_tokenizer,
    hardprefix_bge_model,
    batch_size=64,
    max_len=256,
    desc='Encoding dev claims with hardprefix_finetuned_bge'
)

hardprefix_finetuned_bge_test_emb = encode_bge_texts(
    build_query_inputs(test_texts_raw, use_prefix=True),
    hardprefix_bge_tokenizer,
    hardprefix_bge_model,
    batch_size=64,
    max_len=256,
    desc='Encoding test claims with hardprefix_finetuned_bge'
)

Encoding evidence with hardprefix_finetuned_bge: 100%|██████████| 18888/18888 [32:31<00:00,  9.68it/s]
Encoding train claims with hardprefix_finetuned_bge: 100%|██████████| 20/20 [00:01<00:00, 11.72it/s]
Encoding dev claims with hardprefix_finetuned_bge: 100%|██████████| 3/3 [00:00<00:00, 12.27it/s]
Encoding test claims with hardprefix_finetuned_bge: 100%|██████████| 3/3 [00:00<00:00, 12.43it/s]


In [17]:
# ============================================================
# 2.5 Top-5 retrieval after hard-prefix BGE fine-tuning
# Evaluate on dev set
# ============================================================

predict_dev_hardprefix_finetuned_bge_top5 = dense_retrieve_from_embeddings(
    dev_ids,
    hardprefix_finetuned_bge_dev_emb,
    evidence_ids,
    hardprefix_finetuned_bge_evidence_emb,
    top_k=5,
    batch_size=64
)

print('Hard-prefix fine-tuned BGE dev top-5 metrics:')
hardprefix_finetuned_bge_dev_top5_metrics = eval_retrieval(
    dev_claims,
    predict_dev_hardprefix_finetuned_bge_top5
)

Dense retrieving top-5: 100%|██████████| 3/3 [00:00<00:00, 22.45it/s]

Hard-prefix fine-tuned BGE dev top-5 metrics:
{
  "mean_recall": 0.28863636363636364,
  "mean_precision": 0.16623376623376623,
  "mean_f1": 0.19656771799628944,
  "min_recall": 0.0
}


In [18]:
# ============================================================
# 3.1 SoftPromptBGE wrapper
# Base BGE is frozen; only soft prompt weights are trained.
# ============================================================

import torch.nn as nn

class SoftPromptBGE(nn.Module):
    def __init__(self, base_model, soft_prompt_len=8):
        super().__init__()

        self.base_model = base_model
        self.soft_prompt_len = soft_prompt_len

        hidden_size = base_model.config.hidden_size
        self.soft_prompt = nn.Parameter(
            torch.randn(soft_prompt_len, hidden_size) * 0.02
        )

        for param in self.base_model.parameters():
            param.requires_grad = False

    def encode_queries(self, texts, tokenizer, max_len=256):
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )

        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        token_embeds = self.base_model.embeddings.word_embeddings(input_ids)
        batch_size = token_embeds.size(0)

        soft_prompt_batch = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        inputs_embeds = torch.cat([soft_prompt_batch, token_embeds], dim=1)

        soft_attention = torch.ones(
            batch_size,
            self.soft_prompt_len,
            device=device,
            dtype=attention_mask.dtype
        )
        attention_mask = torch.cat([soft_attention, attention_mask], dim=1)

        outputs = self.base_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )

        # The original CLS token is shifted right by soft_prompt_len.
        embeddings = outputs.last_hidden_state[:, self.soft_prompt_len]
        embeddings = F.normalize(embeddings, p=2, dim=1)

        return embeddings


@torch.no_grad()
def encode_queries_with_soft_prompt(
    texts,
    soft_model,
    tokenizer,
    batch_size=64,
    max_len=256,
    desc='Soft prompt query encoding'
):
    soft_model.eval()
    all_embeddings = []

    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch_texts = texts[start:start + batch_size]
        embeddings = soft_model.encode_queries(
            batch_texts,
            tokenizer,
            max_len=max_len
        )
        all_embeddings.append(embeddings.detach().cpu())

    return torch.cat(all_embeddings, dim=0)

In [19]:
# ============================================================
# 3.2 Soft prompt tuning on top of hard-prefix fine-tuned BGE
# Train set only
# ============================================================

# Freeze the already fine-tuned BGE weights.
soft_hardprefix_bge_base_model = hardprefix_bge_model
soft_hardprefix_bge_tokenizer = hardprefix_bge_tokenizer

soft_hardprefix_bge_model = SoftPromptBGE(
    base_model=soft_hardprefix_bge_base_model,
    soft_prompt_len=8
).to(device)

SOFT_PROMPT_BATCH_SIZE = 16
SOFT_PROMPT_EPOCHS = 3
SOFT_PROMPT_LR = 5e-3
SOFT_PROMPT_TEMPERATURE = 0.05

soft_prompt_train_loader = make_pair_loader(
    train_pairs_with_prefix,
    batch_size=SOFT_PROMPT_BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

soft_prompt_optimizer = AdamW(
    [soft_hardprefix_bge_model.soft_prompt],
    lr=SOFT_PROMPT_LR
)

for epoch in range(SOFT_PROMPT_EPOCHS):
    soft_hardprefix_bge_model.train()
    epoch_losses = []

    for batch in tqdm(
        soft_prompt_train_loader,
        desc=f'Soft prompt tuning epoch {epoch + 1}/{SOFT_PROMPT_EPOCHS}'
    ):
        query_emb = soft_hardprefix_bge_model.encode_queries(
            batch['queries'],
            soft_hardprefix_bge_tokenizer,
            max_len=256
        )

        with torch.no_grad():
            positive_emb = encode_for_finetuning(
                batch['positives'],
                soft_hardprefix_bge_tokenizer,
                soft_hardprefix_bge_base_model,
                max_len=512
            )

        loss = in_batch_contrastive_loss(
            query_emb,
            positive_emb,
            temperature=SOFT_PROMPT_TEMPERATURE
        )

        soft_prompt_optimizer.zero_grad()
        loss.backward()
        soft_prompt_optimizer.step()

        epoch_losses.append(loss.item())

    print(f'Epoch {epoch + 1} mean loss: {np.mean(epoch_losses):.6f}')

# Important variable names:
# soft_hardprefix_bge_model
# soft_hardprefix_bge_model.soft_prompt

Soft prompt tuning epoch 1/3: 100%|██████████| 258/258 [00:19<00:00, 13.40it/s]


Epoch 1 mean loss: 0.341479


Soft prompt tuning epoch 2/3: 100%|██████████| 258/258 [00:19<00:00, 13.47it/s]


Epoch 2 mean loss: 0.353693


Soft prompt tuning epoch 3/3: 100%|██████████| 258/258 [00:19<00:00, 13.49it/s]

Epoch 3 mean loss: 0.355051


In [20]:
# ============================================================
# 3.3 Encode claim sets with soft prompt + hard prefix + fine-tuned BGE
# Evidence embeddings still use the hard-prefix fine-tuned base BGE.
# ============================================================

soft_hardprefix_finetuned_bge_evidence_emb = hardprefix_finetuned_bge_evidence_emb

soft_hardprefix_finetuned_bge_train_emb = encode_queries_with_soft_prompt(
    build_query_inputs(train_texts_raw, use_prefix=True),
    soft_hardprefix_bge_model,
    soft_hardprefix_bge_tokenizer,
    batch_size=64,
    max_len=256,
    desc='Encoding train claims with soft+hardprefix+finetuned BGE'
)

soft_hardprefix_finetuned_bge_dev_emb = encode_queries_with_soft_prompt(
    build_query_inputs(dev_texts_raw, use_prefix=True),
    soft_hardprefix_bge_model,
    soft_hardprefix_bge_tokenizer,
    batch_size=64,
    max_len=256,
    desc='Encoding dev claims with soft+hardprefix+finetuned BGE'
)

soft_hardprefix_finetuned_bge_test_emb = encode_queries_with_soft_prompt(
    build_query_inputs(test_texts_raw, use_prefix=True),
    soft_hardprefix_bge_model,
    soft_hardprefix_bge_tokenizer,
    batch_size=64,
    max_len=256,
    desc='Encoding test claims with soft+hardprefix+finetuned BGE'
)

Encoding train claims with soft+hardprefix+finetuned BGE: 100%|██████████| 20/20 [00:01<00:00, 10.24it/s]
Encoding dev claims with soft+hardprefix+finetuned BGE: 100%|██████████| 3/3 [00:00<00:00, 11.33it/s]
Encoding test claims with soft+hardprefix+finetuned BGE: 100%|██████████| 3/3 [00:00<00:00, 11.10it/s]


In [21]:
# ============================================================
# 3.4 Top-5 check after soft prompt tuning
# Evaluate on dev set
# ============================================================

predict_dev_soft_hardprefix_finetuned_bge_top5 = dense_retrieve_from_embeddings(
    dev_ids,
    soft_hardprefix_finetuned_bge_dev_emb,
    evidence_ids,
    soft_hardprefix_finetuned_bge_evidence_emb,
    top_k=5,
    batch_size=64
)

print('Soft prompt + hard-prefix + fine-tuned BGE dev top-5 metrics:')
soft_hardprefix_finetuned_bge_dev_top5_metrics = eval_retrieval(
    dev_claims,
    predict_dev_soft_hardprefix_finetuned_bge_top5
)

Dense retrieving top-5: 100%|██████████| 3/3 [00:00<00:00, 33.07it/s]

Soft prompt + hard-prefix + fine-tuned BGE dev top-5 metrics:
{
  "mean_recall": 0.2857142857142857,
  "mean_precision": 0.16363636363636364,
  "mean_f1": 0.1938260152545867,
  "min_recall": 0.0
}


In [24]:
# ============================================================
# 4.0 Clear GPU memory before BM25S
# BM25S does not need GPU. Force JAX/BM25S to use CPU.
# ============================================================

import os
import gc
import torch

# ------------------------------------------------------------
# 1. Force JAX to use CPU before importing bm25s
# ------------------------------------------------------------
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["JAX_PLATFORMS"] = "cpu"

# ------------------------------------------------------------
# 2. Delete heavy training objects if they exist
# These are no longer needed after embeddings / predictions are saved
# ------------------------------------------------------------
objects_to_delete = [
    "trainer",
    "train_result",
    "model",
    "bge_model",
    "base_bge_model",
    "finetuned_bge_model",
    "soft_prompt_model",
    "cross_encoder_model",
    "optimizer",
    "scheduler",
    "train_dataloader",
    "dev_dataloader",
    "test_dataloader",
    "train_dataset",
    "dev_dataset",
    "test_dataset",
]

for name in objects_to_delete:
    if name in globals():
        del globals()[name]

# ------------------------------------------------------------
# 3. Move embedding tensors from GPU to CPU
# Keep the variables, but remove them from GPU memory
# ------------------------------------------------------------
for name, value in list(globals().items()):
    if torch.is_tensor(value):
        if value.is_cuda:
            globals()[name] = value.detach().cpu()
            print(f"Moved tensor to CPU: {name}")

# ------------------------------------------------------------
# 4. Clear PyTorch GPU cache
# ------------------------------------------------------------
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"GPU allocated after cleanup: {allocated:.2f} GB")
    print(f"GPU reserved after cleanup:   {reserved:.2f} GB")

print("GPU memory cleanup finished. Now run the BM25S cell.")

Moved tensor to CPU: query_emb
Moved tensor to CPU: positive_emb
Moved tensor to CPU: loss
GPU allocated after cleanup: 0.53 GB
GPU reserved after cleanup:   0.66 GB
GPU memory cleanup finished. Now run the BM25S cell.


In [25]:
# ============================================================
# 4.1 BM25S indexing and retrieval functions
# ============================================================

import bm25s

def build_bm25s_retriever(
    evidence_ids,
    evidence_dataset,
    k1=1.5,
    b=0.75
):
    evidence_texts_for_bm25 = [
        evidence_dataset[evidence_id]
        for evidence_id in evidence_ids
    ]

    corpus_tokens = bm25s.tokenize(
        evidence_texts_for_bm25,
        stopwords='en'
    )

    bm25s_retriever = bm25s.BM25(
        k1=k1,
        b=b
    )

    bm25s_retriever.index(corpus_tokens)

    return bm25s_retriever


def bm25s_retrieve_topk(
    retriever,
    claim_ids,
    claims_dataset,
    evidence_ids,
    top_k=100
):
    query_texts = [
        claims_dataset[claim_id]['claim_text']
        for claim_id in claim_ids
    ]

    query_tokens = bm25s.tokenize(
        query_texts,
        stopwords='en'
    )

    retrieved_docs, retrieved_scores = retriever.retrieve(
        query_tokens,
        corpus=evidence_ids,
        k=top_k
    )

    predictions = {}

    for row_index, claim_id in enumerate(claim_ids):
        predictions[claim_id] = [
            str(evidence_id)
            for evidence_id in retrieved_docs[row_index]
        ]

    return predictions


print('BM25S helper functions ready.')

BM25S helper functions ready.


In [26]:
# ============================================================
# 4.2 Tune BM25S k1 and b on train set, using k=5
# ============================================================

bm25s_search_space = []

for k1_value in [0.6, 0.8, 1.0]:
    for b_value in [0.25, 0.5, 0.75, 1.0]:
        bm25s_search_space.append((k1_value, b_value))

best_bm25s_train_top5_f1 = -1.0
best_bm25s_k1 = None
best_bm25s_b = None
best_bm25s_retriever = None
best_bm25s_train_top5_metrics = None
predict_train_bm25s_top5 = None

for k1_value, b_value in tqdm(
    bm25s_search_space,
    desc='Tuning BM25S on train set with top-5'
):
    temp_retriever = build_bm25s_retriever(
        evidence_ids=evidence_ids,
        evidence_dataset=evidence,
        k1=k1_value,
        b=b_value
    )

    temp_train_top5 = bm25s_retrieve_topk(
        retriever=temp_retriever,
        claim_ids=train_ids,
        claims_dataset=train_claims,
        evidence_ids=evidence_ids,
        top_k=5
    )

    temp_metrics = eval_retrieval(
        train_claims,
        temp_train_top5
    )

    temp_f1 = temp_metrics['mean_f1']

    if temp_f1 > best_bm25s_train_top5_f1:
        best_bm25s_train_top5_f1 = temp_f1
        best_bm25s_k1 = k1_value
        best_bm25s_b = b_value
        best_bm25s_retriever = temp_retriever
        best_bm25s_train_top5_metrics = temp_metrics
        predict_train_bm25s_top5 = temp_train_top5

print('Best BM25S k1:', best_bm25s_k1)
print('Best BM25S b:', best_bm25s_b)
print('Best BM25S train top-5 metrics:')
best_bm25s_train_top5_metrics

Tuning BM25S on train set with top-5:   0%|          | 0/12 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1208827 [00:00<?, ?it/s]

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/1208827 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/1208827 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1228 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1228 [00:00<?, ?it/s]

Tuning BM25S on train set with top-5: 100%|██████████| 12/12 [13:33<00:00, 67.78s/it]

{
  "mean_recall": 0.13990228013029316,
  "mean_precision": 0.08713355048859935,
  "mean_f1": 0.10145739620495321,
  "min_recall": 0.0
}
Best BM25S k1: 0.6
Best BM25S b: 0.5
Best BM25S train top-5 metrics:


{'mean_recall': 0.16764386536373507,
 'mean_precision': 0.10228013029315962,
 'mean_f1': 0.11957887389483482,
 'min_recall': 0.0}

In [27]:
# ============================================================
# 5.1 Dense top-100 candidates from soft+hardprefix+finetuned BGE
# ============================================================

predict_train_soft_hardprefix_finetuned_bge_top100 = dense_retrieve_from_embeddings(
    train_ids,
    soft_hardprefix_finetuned_bge_train_emb,
    evidence_ids,
    soft_hardprefix_finetuned_bge_evidence_emb,
    top_k=100,
    batch_size=64
)

predict_dev_soft_hardprefix_finetuned_bge_top100 = dense_retrieve_from_embeddings(
    dev_ids,
    soft_hardprefix_finetuned_bge_dev_emb,
    evidence_ids,
    soft_hardprefix_finetuned_bge_evidence_emb,
    top_k=100,
    batch_size=64
)

predict_test_soft_hardprefix_finetuned_bge_top100 = dense_retrieve_from_embeddings(
    test_ids,
    soft_hardprefix_finetuned_bge_test_emb,
    evidence_ids,
    soft_hardprefix_finetuned_bge_evidence_emb,
    top_k=100,
    batch_size=64
)

Dense retrieving top-100: 100%|██████████| 3/3 [00:00<00:00, 31.07it/s]


In [28]:
# ============================================================
# 5.2 BM25S top-100 candidates using best BM25S
# ============================================================

predict_train_bm25s_top100 = bm25s_retrieve_topk(
    retriever=best_bm25s_retriever,
    claim_ids=train_ids,
    claims_dataset=train_claims,
    evidence_ids=evidence_ids,
    top_k=100
)

predict_dev_bm25s_top100 = bm25s_retrieve_topk(
    retriever=best_bm25s_retriever,
    claim_ids=dev_ids,
    claims_dataset=dev_claims,
    evidence_ids=evidence_ids,
    top_k=100
)

predict_test_bm25s_top100 = bm25s_retrieve_topk(
    retriever=best_bm25s_retriever,
    claim_ids=test_ids,
    claims_dataset=test_claims,
    evidence_ids=evidence_ids,
    top_k=100
)

Split strings:   0%|          | 0/1228 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1228 [00:00<?, ?it/s]

Split strings:   0%|          | 0/154 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/154 [00:00<?, ?it/s]

Split strings:   0%|          | 0/153 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/153 [00:00<?, ?it/s]

In [29]:
# ============================================================
# 5.3 Union dense top-100 + BM25S top-100 candidates
# ============================================================

predict_train_final_candidate_top100_union = union_candidate_lists(
    predict_train_soft_hardprefix_finetuned_bge_top100,
    predict_train_bm25s_top100
)

predict_dev_final_candidate_top100_union = union_candidate_lists(
    predict_dev_soft_hardprefix_finetuned_bge_top100,
    predict_dev_bm25s_top100
)

predict_test_final_candidate_top100_union = union_candidate_lists(
    predict_test_soft_hardprefix_finetuned_bge_top100,
    predict_test_bm25s_top100
)

print('Example train candidate count:', len(next(iter(predict_train_final_candidate_top100_union.values()))))
print('Example dev candidate count:', len(next(iter(predict_dev_final_candidate_top100_union.values()))))
print('Example test candidate count:', len(next(iter(predict_test_final_candidate_top100_union.values()))))

Example train candidate count: 182
Example dev candidate count: 183
Example test candidate count: 187


In [30]:
# ============================================================
# 5.4 Build CrossEncoder training examples from train candidates
# Positives: gold evidences.
# Negatives: candidate evidences not in gold labels.
# ============================================================

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

def build_cross_encoder_train_examples(
    train_claims,
    evidence,
    candidate_predictions,
    max_negatives_per_claim=5
):
    train_examples = []

    rng = np.random.default_rng(SEED)

    for claim_id, claim in tqdm(
        train_claims.items(),
        desc='Building CrossEncoder train examples'
    ):
        claim_text = claim['claim_text']
        gold_evidence_ids = [
            eid
            for eid in claim.get('evidences', [])
            if eid in evidence
        ]

        gold_set = set(gold_evidence_ids)

        for evidence_id in gold_evidence_ids:
            train_examples.append(
                InputExample(
                    texts=[claim_text, evidence[evidence_id]],
                    label=1.0
                )
            )

        negative_ids = [
            eid
            for eid in candidate_predictions.get(claim_id, [])
            if eid in evidence and eid not in gold_set
        ]

        if len(negative_ids) > max_negatives_per_claim:
            negative_ids = list(
                rng.choice(
                    negative_ids,
                    size=max_negatives_per_claim,
                    replace=False
                )
            )

        for evidence_id in negative_ids:
            train_examples.append(
                InputExample(
                    texts=[claim_text, evidence[evidence_id]],
                    label=0.0
                )
            )

    return train_examples


cross_encoder_train_examples = build_cross_encoder_train_examples(
    train_claims=train_claims,
    evidence=evidence,
    candidate_predictions=predict_train_final_candidate_top100_union,
    max_negatives_per_claim=5
)

print('CrossEncoder train examples:', len(cross_encoder_train_examples))
print(cross_encoder_train_examples[0].texts, cross_encoder_train_examples[0].label)

Building CrossEncoder train examples: 100%|██████████| 1228/1228 [00:00<00:00, 3937.97it/s]

CrossEncoder train examples: 10262
['Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.', 'At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.'] 1.0


In [31]:
# ============================================================
# 5.5 Fine-tune CrossEncoder on train set
# ============================================================

CROSS_ENCODER_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

finetuned_cross_encoder_model = CrossEncoder(
    CROSS_ENCODER_MODEL_NAME,
    num_labels=1,
    device=str(device)
)

CROSS_ENCODER_BATCH_SIZE = 16
CROSS_ENCODER_EPOCHS = 2

cross_encoder_train_dataloader = DataLoader(
    cross_encoder_train_examples,
    shuffle=True,
    batch_size=CROSS_ENCODER_BATCH_SIZE
)

cross_encoder_warmup_steps = int(
    len(cross_encoder_train_dataloader) * CROSS_ENCODER_EPOCHS * 0.1
)

finetuned_cross_encoder_model.fit(
    train_dataloader=cross_encoder_train_dataloader,
    epochs=CROSS_ENCODER_EPOCHS,
    warmup_steps=cross_encoder_warmup_steps,
    show_progress_bar=True
)

# Important variable name:
# finetuned_cross_encoder_model

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Step,Training Loss
500,0.696200
1000,0.487392


In [32]:
# ============================================================
# 5.6 CrossEncoder reranking function
# ============================================================

def rerank_with_cross_encoder(
    claims_dataset,
    evidence_dataset,
    candidate_predictions,
    cross_encoder_model,
    top_k=5,
    batch_size=32
):
    reranked_predictions = {}

    for claim_id, candidate_evidence_ids in tqdm(
        candidate_predictions.items(),
        desc=f'CrossEncoder reranking top-{top_k}'
    ):
        claim_text = claims_dataset[claim_id]['claim_text']

        pairs = []
        valid_evidence_ids = []

        for evidence_id in candidate_evidence_ids:
            if evidence_id in evidence_dataset:
                pairs.append([claim_text, evidence_dataset[evidence_id]])
                valid_evidence_ids.append(evidence_id)

        if len(pairs) == 0:
            reranked_predictions[claim_id] = []
            continue

        scores = cross_encoder_model.predict(
            pairs,
            batch_size=batch_size,
            show_progress_bar=False
        )

        ranked_indices = np.argsort(scores)[::-1]

        reranked_predictions[claim_id] = [
            valid_evidence_ids[index]
            for index in ranked_indices[:top_k]
        ]

    return reranked_predictions

In [33]:
# ============================================================
# 6.1 Final top-5 predictions on train and dev
# Full model:
# soft prompt + hard prefix + BGE fine-tuned top-100
# + BM25S tuned top-100
# + fine-tuned CrossEncoder reranking top-5
# ============================================================

predict_train_final_crossencoder_top5 = rerank_with_cross_encoder(
    claims_dataset=train_claims,
    evidence_dataset=evidence,
    candidate_predictions=predict_train_final_candidate_top100_union,
    cross_encoder_model=finetuned_cross_encoder_model,
    top_k=5,
    batch_size=32
)

predict_dev_final_crossencoder_top5 = rerank_with_cross_encoder(
    claims_dataset=dev_claims,
    evidence_dataset=evidence,
    candidate_predictions=predict_dev_final_candidate_top100_union,
    cross_encoder_model=finetuned_cross_encoder_model,
    top_k=5,
    batch_size=32
)

print('Final train top-5 metrics:')
final_train_top5_metrics = eval_retrieval(
    train_claims,
    predict_train_final_crossencoder_top5
)

print('Final dev top-5 metrics:')
final_dev_top5_metrics = eval_retrieval(
    dev_claims,
    predict_dev_final_crossencoder_top5
)

CrossEncoder reranking top-5: 100%|██████████| 154/154 [00:23<00:00,  6.52it/s]

Final train top-5 metrics:
{
  "mean_recall": 0.3269001085776331,
  "mean_precision": 0.19918566775244298,
  "mean_f1": 0.23297722454888586,
  "min_recall": 0.0
}
Final dev top-5 metrics:
{
  "mean_recall": 0.34177489177489184,
  "mean_precision": 0.19740259740259739,
  "mean_f1": 0.23421974850546284,
  "min_recall": 0.0
}


In [34]:
# ============================================================
# 7.1 Final top-5 predictions on test set
# ============================================================

predict_test_final_crossencoder_top5 = rerank_with_cross_encoder(
    claims_dataset=test_claims,
    evidence_dataset=evidence,
    candidate_predictions=predict_test_final_candidate_top100_union,
    cross_encoder_model=finetuned_cross_encoder_model,
    top_k=5,
    batch_size=32
)

print('Number of test predictions:', len(predict_test_final_crossencoder_top5))

# This is the final test prediction variable:
# predict_test_final_crossencoder_top5

example_test_claim_id = test_ids[0]
print('Example test claim id:', example_test_claim_id)
print('Predicted evidence:', predict_test_final_crossencoder_top5[example_test_claim_id])

CrossEncoder reranking top-5: 100%|██████████| 153/153 [00:23<00:00,  6.56it/s]

Number of test predictions: 153
Example test claim id: claim-2967
Predicted evidence: ['evidence-308923', 'evidence-219780', 'evidence-963856', 'evidence-632574', 'evidence-905191']


In [35]:
# ============================================================
# Inspect first 3 elements of final prediction dictionaries
# ============================================================

def inspect_first_three_predictions(predictions, claims_dataset, evidence_dataset, set_name):
    print("=" * 80)
    print(f"{set_name}: first 3 claim -> predicted evidence results")
    print("=" * 80)

    first_three_items = list(predictions.items())[:3]

    for i, (claim_id, predicted_evidence_ids) in enumerate(first_three_items, start=1):
        print(f"\n[{i}] Claim ID: {claim_id}")
        print("Claim text:")
        print(claims_dataset[claim_id]["claim_text"])

        print("\nPredicted evidence IDs:")
        print(predicted_evidence_ids)

        print("\nPredicted evidence text:")
        for rank, evidence_id in enumerate(predicted_evidence_ids, start=1):
            print(f"\nTop {rank} Evidence ID: {evidence_id}")
            print(evidence_dataset.get(evidence_id, "[Evidence ID not found]"))

        print("\n" + "-" * 80)


inspect_first_three_predictions(
    predictions=predict_train_final_crossencoder_top5,
    claims_dataset=train_claims,
    evidence_dataset=evidence,
    set_name="TRAIN"
)

inspect_first_three_predictions(
    predictions=predict_dev_final_crossencoder_top5,
    claims_dataset=dev_claims,
    evidence_dataset=evidence,
    set_name="DEV"
)

inspect_first_three_predictions(
    predictions=predict_test_final_crossencoder_top5,
    claims_dataset=test_claims,
    evidence_dataset=evidence,
    set_name="TEST"
)

TRAIN: first 3 claim -> predicted evidence results

[1] Claim ID: claim-2244
Claim text:
So CO2 causes warming AND rising temperature causes CO2 rise.

Predicted evidence IDs:
['evidence-368192', 'evidence-548766', 'evidence-843608', 'evidence-178206', 'evidence-625563']

Predicted evidence text:

Top 1 Evidence ID: evidence-368192
Increases in atmospheric concentrations of CO 2 and other long-lived greenhouse gases such as methane, nitrous oxide and ozone have correspondingly strengthened their absorption and emission of infrared radiation, causing the rise in average global temperature since the mid-20th century.

Top 2 Evidence ID: evidence-548766
Due to the increase in temperature of the soil, CO2 levels in our atmosphere increase, and as such the mean average temperature of the Earth is rising.

Top 3 Evidence ID: evidence-843608
Following the start of the Industrial Revolution, atmospheric CO 2 concentration increased to over 400 parts per million and continues to increase, causi

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*